# Data

## CFB API Data

### Recruiting Rankings

In [2]:
import requests
import pandas as pd
import os
import time

os.makedirs("data", exist_ok=True)
api_key = "dtZVj1XqHVj5dfXxdc/nqukfZgSAP6wvRvPIHIMWpSTqSrMzmfzoxVJeCdMi2/Jo"
headers = {"Authorization": f"Bearer {api_key}"}
url = "https://api.collegefootballdata.com/recruiting/teams"

def pull_yearly(url, start=2005, end=2024, label=""):
    dfs = []
    for year in range(start, end):
        r = requests.get(url, headers=headers, params={"year": year})
        data = r.json()
        if isinstance(data, list) and len(data) > 0:
            dfs.append(pd.DataFrame(data))
        else:
            print(f"  No data for year {year}")
        time.sleep(0.2)
    result = pd.concat(dfs, ignore_index=True)
    print(f"{label}: {result.shape}")
    return result

# Recruiting
print("Pulling recruiting...")
recruiting_df = pull_yearly(
    "https://api.collegefootballdata.com/recruiting/teams",
    label="Recruiting"
)
recruiting_df.to_csv("data/recruiting.csv", index=False)

# SP+ ratings
print("Pulling SP+ ratings...")
sp_df = pull_yearly(
    "https://api.collegefootballdata.com/ratings/sp",
    label="SP+"
)
sp_df.to_csv("data/sp_ratings.csv", index=False)

# Win/loss records
print("Pulling records...")
records_df = pull_yearly(
    "https://api.collegefootballdata.com/records",
    label="Records"
)
records_df.to_csv("data/records.csv", index=False)

# Talent composite
print("Pulling talent...")
talent_df = pull_yearly(
    "https://api.collegefootballdata.com/talent",
    label="Talent"
)
talent_df.to_csv("data/talent.csv", index=False)

# Coaches
print("Pulling coaches...")
coaches_df = pull_yearly(
    "https://api.collegefootballdata.com/coaches",
    label="Coaches"
)
coaches_df.to_csv("data/coaches.csv", index=False)

print("\nAll done — files saved to data/")

Pulling recruiting...
Recruiting: (3388, 4)
Pulling SP+ ratings...


KeyboardInterrupt: 

## Checking Data

In [3]:
import pandas as pd

files = {
    "recruiting": "data/recruiting.csv",
    "sp_ratings": "data/sp_ratings.csv",
    "records": "data/records.csv",
    "talent": "data/talent.csv",
    "coaches": "data/coaches.csv"
}

for name, path in files.items():
    df = pd.read_csv(path)
    print(f"{name.upper()}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    print(df.head(3).to_string())
    print()

RECRUITING
Shape: (3388, 4)
Columns: ['year', 'team', 'rank', 'points']
Missing values:
Series([], dtype: int64)

Sample rows:
   year           team  rank  points
0  2005            USC     1  285.12
1  2005  Florida State     2  281.52
2  2005       Oklahoma     3  279.20

SP_RATINGS
Shape: (2401, 10)
Columns: ['year', 'team', 'conference', 'rating', 'ranking', 'secondOrderWins', 'sos', 'offense', 'defense', 'specialTeams']
Missing values:
conference          19
ranking             19
secondOrderWins    659
sos                659
dtype: int64

Sample rows:
   year        team conference  rating  ranking  secondOrderWins    sos                                                                                                                                                                                      offense                                                                                                                                                                               

Recruiting Data: clean, no issues so far
SP+: the offense, defense, and specialTeams columns are nested as strings
Records: total, conferenceGames... are all nested dicts. Need to parse total. Also includes FCS teams so I'll have to filter to classification == 'fbs'
Coaches: seasons column is a nested list of dicts

In [4]:
import pandas as pd
import ast

# recruiting (already clean)
recruiting = pd.read_csv("data/recruiting.csv")

# SP+ (extracting only what is needed)
sp = pd.read_csv("data/sp_ratings.csv")
sp = sp[["year", "team", "conference", "rating", "ranking"]].copy()
sp.rename(columns={"rating": "sp_rating", "ranking": "sp_ranking"}, inplace=True)

# Records (parse nested total dict, filter to FBS) 
records_raw = pd.read_csv("data/records.csv")
records_raw = records_raw[records_raw["classification"] == "fbs"].copy()

def parse_total(val):
    try:
        d = ast.literal_eval(val)
        return d.get("wins"), d.get("losses"), d.get("games")
    except:
        return None, None, None

records_raw[["wins", "losses", "games"]] = records_raw["total"].apply(
    lambda x: pd.Series(parse_total(x))
)
records = records_raw[["year", "team", "conference", "wins", "losses", "games"]].copy()

# talent (already clean)
talent = pd.read_csv("data/talent.csv")

# coaches (explode seasons list into one row per coach-season)
coaches_raw = pd.read_csv("data/coaches.csv")

rows = []
for _, row in coaches_raw.iterrows():
    try:
        seasons = ast.literal_eval(row["seasons"])
        for s in seasons:
            rows.append({
                "first_name": row["firstName"],
                "last_name": row["lastName"],
                "school": s.get("school"),
                "year": s.get("year"),
                "wins": s.get("wins"),
                "losses": s.get("losses"),
                "sp_overall": s.get("spOverall"),
                "sp_offense": s.get("spOffense"),
                "sp_defense": s.get("spDefense"),
            })
    except:
        continue

coaches = pd.DataFrame(rows)
coaches["coach_name"] = coaches["first_name"] + " " + coaches["last_name"]

# check
for name, df in [("recruiting", recruiting), ("sp", sp), 
                  ("records", records), ("talent", talent), 
                  ("coaches", coaches)]:
    print(f"{name}: {df.shape}")
    print(df.head(2).to_string())
    print()

# saving cleaned versions of data
recruiting.to_csv("data/recruiting_clean.csv", index=False)
sp.to_csv("data/sp_clean.csv", index=False)
records.to_csv("data/records_clean.csv", index=False)
talent.to_csv("data/talent_clean.csv", index=False)
coaches.to_csv("data/coaches_clean.csv", index=False)

print("All cleaned files saved.")

recruiting: (3388, 4)
   year           team  rank  points
0  2005            USC     1  285.12
1  2005  Florida State     2  281.52

sp: (2401, 5)
   year   team conference  sp_rating  sp_ranking
0  2005  Texas        SEC       35.3         1.0
1  2005    USC    Big Ten       34.7         2.0

records: (2383, 6)
   year    team      conference  wins  losses  games
0  2005  Auburn             SEC     9       3     12
1  2005     UAB  Conference USA     5       6     11

talent: (2010, 3)
   year     team  talent
0  2015  Alabama  981.90
1  2015      USC  926.71

coaches: (2495, 10)
  first_name last_name     school  year  wins  losses  sp_overall  sp_offense  sp_defense     coach_name
0      Barry   Alvarez  Wisconsin  2005    10       3        11.1        35.3        24.2  Barry Alvarez
1      Chuck     Amato   NC State  2005     7       5         9.4        22.1        12.7    Chuck Amato

All cleaned files saved.


Only issue: 2,495 rows across the 19 years is only 131 coaches per year. Could potentially have gaps

### School Name Standardization

In [5]:


# Load cleaned data
recruiting = pd.read_csv("data/recruiting_clean.csv")
draft = pd.read_csv("data/nfl_draft_prospects.csv")

# Filter draft to 2005+ and actual drafted players
draft = draft[(draft["draft_year"] >= 2005) & (draft["round"].notna())].copy()

# Get unique school names from each
cfbd_teams = set(recruiting["team"].dropna().str.strip().unique())
draft_schools = set(draft["school"].dropna().str.strip().unique())

# Find mismatches
only_in_draft = sorted(draft_schools - cfbd_teams)
only_in_cfbd = sorted(cfbd_teams - draft_schools)
in_both = sorted(draft_schools & cfbd_teams)

print(f"Schools in both: {len(in_both)}")
print(f"Schools only in draft (need mapping): {len(only_in_draft)}")
print(f"Schools only in CFBD (no draft picks or name diff): {len(only_in_cfbd)}")
print()
print("=== DRAFT SCHOOLS NOT IN CFBD ===")
for s in only_in_draft:
    print(f"  '{s}'")
print()
print("=== CFBD SCHOOLS NOT IN DRAFT ===")
for s in only_in_cfbd:
    print(f"  '{s}'")

Schools in both: 205
Schools only in draft (need mapping): 87
Schools only in CFBD (no draft picks or name diff): 50

=== DRAFT SCHOOLS NOT IN CFBD ===
  'Abilene Christian'
  'Albany'
  'Albany State (GA)'
  'Albion'
  'Appalachian State'
  'Ashland'
  'Australia'
  'Bentley'
  'Bethel (TN)'
  'Bloomsburg'
  'California (PA)'
  'Central Missouri'
  'Central Missouri State'
  'Chadron State'
  'Charleston (WV)'
  'Colorado State-Pueblo'
  'Concordia-St. Paul'
  'Connecticut'
  'East Central'
  'Ferris State'
  'Florida Intl'
  'Fort Hays State'
  'Grambling State'
  'Grand Valley State'
  'Harding'
  'Hillsdale'
  'Hobart'
  'Hofstra'
  'Humboldt State'
  'Indiana (PA)'
  'Kutztown'
  'Lane'
  'Lenoir-Rhyne'
  'Lindenwood'
  'Louisiana Monroe'
  'Manitoba'
  'Mars Hill'
  'McGill'
  'McNeese State'
  'Miami (FL)'
  'Michigan Tech'
  'Midwestern State'
  'Missouri Southern State'
  'Missouri Western'
  'Morehouse'
  'Mount Union'
  'Nebraska-Omaha'
  'Newberry'
  'Nicholls State'
  'Nor

Most mismatches are abbreviations, name variations, and FCS schools 

In [6]:
# Schools in draft that need mapping to CFBD names
# None = drop (FCS, D2, D3, international — not in FBS recruiting data)
draft_to_cfbd = {
    # Clear name fixes — these are FBS schools
    "Appalachian State":        "App State",
    "Connecticut":              "UConn",
    "Florida Intl":             "FIU",
    "Louisiana Monroe":         "Louisiana-Monroe",
    "San Jose State":           "San José State",
    "Southern Mississippi":     "Southern Miss",
    "Stephen F Austin":         "Stephen F. Austin",
    "UMass":                    "Massachusetts",
    "UT San Antonio":           "UTSA",
    "Tennessee-Martin":         "UT Martin",
    "Sam Houston State":        "Sam Houston",
    "North Alabama":            "North Alabama",
    "Tarleton State":           "Tarleton State",

    # FCS schools — keep if they have draft picks but won't merge to recruiting
    # These will be dropped in the merge since they're not in FBS recruiting
    "Abilene Christian":        None,
    "Albany State (GA)":        None,
    "Albion":                   None,
    "Ashland":                  None,
    "Australia":                None,
    "Bethel (TN)":              None,
    "Bloomsburg":               None,
    "California (PA)":          None,
    "Central Missouri":         None,
    "Chadron State":            None,
    "Charleston (WV)":          None,
    "Colorado State-Pueblo":    None,
    "Concordia-St. Paul":       None,
    "East Central":             None,
    "Ferris State":             None,
    "Fort Hays State":          None,
    "Grand Valley State":       None,
    "Harding":                  None,
    "Hillsdale":                None,
    "Hobart":                   None,
    "Humboldt State":           None,
    "Indiana (PA)":             None,
    "Kutztown":                 None,
    "Lenoir-Rhyne":             None,
    "Lindenwood":               None,
    "Manitoba":                 None,
    "Mars Hill":                None,
    "McGill":                   None,
    "Midwestern State":         None,
    "Missouri Southern State":  None,
    "Missouri Western":         None,
    "Morehouse":                None,
    "Mount Union":              None,
    "Newberry":                 None,
    "Northeastern State":       None,
    "Northwest Missouri State": None,
    "Pittsburg State":          None,
    "Prairie View":             "Prairie View A&M",
    "Presbyterian College":     "Presbyterian",
    "Saginaw Valley":           None,
    "Sioux Falls":              None,
    "Southeastern Louisiana":   "SE Louisiana",
    "St. John's (MN)":          None,
    "Valdosta State":           None,
    "Virginia State":           None,
    "Washburn":                 None,
    "West Alabama":             None,
    "West Georgia":             None,
    "West Texas A&M":           None,
    "Western Oregon":           None,
    "Wisconsin-Whitewater":     None,
}

# Apply mapping to draft data
draft["school_std"] = draft["school"].map(
    lambda x: draft_to_cfbd.get(x, x) if x in draft_to_cfbd else x
)

# Drop rows where mapping is None (FCS/D2/international)
draft_fbs = draft[draft["school_std"].notna()].copy()

# Verify overlap now
cfbd_teams = set(recruiting["team"].dropna().str.strip().unique())
draft_schools_std = set(draft_fbs["school_std"].dropna().str.strip().unique())

still_missing = sorted(draft_schools_std - cfbd_teams)
matched = sorted(draft_schools_std & cfbd_teams)

print(f"Matched schools: {len(matched)}")
print(f"Still unmatched after mapping: {len(still_missing)}")
if still_missing:
    print("\nStill unmatched:")
    for s in still_missing:
        print(f"  '{s}'")

print(f"\nDraft picks remaining after FCS drop: {draft_fbs.shape[0]}")
print(f"Draft picks dropped (FCS/D2): {draft.shape[0] - draft_fbs.shape[0]}")

Matched schools: 214
Still unmatched after mapping: 27

Still unmatched:
  'Albany'
  'Bentley'
  'Central Missouri State'
  'FIU'
  'Grambling State'
  'Hofstra'
  'Lane'
  'Louisiana-Monroe'
  'McNeese State'
  'Miami (FL)'
  'Michigan Tech'
  'Nebraska-Omaha'
  'Nicholls State'
  'North Alabama'
  'North Carolina State'
  'St. Augustine's'
  'St. Paul's College'
  'Stillman'
  'Tarleton State'
  'Tuskegee'
  'UW-Whitewater'
  'Western Ontario'
  'Wheaton College (Ill)'
  'Whitworth'
  'William Penn'
  'Wingate'
  'Winston-Salem'

Draft picks remaining after FCS drop: 4246
Draft picks dropped (FCS/D2): 85


Checking what CFBD calls the 4 schools not standardized

In [7]:
search_terms = ["FIU", "Florida International", "Louisiana", "Monroe", 
                "North Alabama", "Tarleton"]

for term in search_terms:
    matches = [t for t in cfbd_teams if term.lower() in t.lower()]
    if matches:
        print(f"Search '{term}': {matches}")

Search 'Florida International': ['Florida International']
Search 'Louisiana': ['Louisiana', 'SE Louisiana', 'Louisiana Tech']
Search 'Monroe': ['UL Monroe']


In [8]:
# Fix the 4 remaining mismatches
draft_to_cfbd["Florida Intl"] = "Florida International"  # update existing entry
draft_to_cfbd["Louisiana Monroe"] = "UL Monroe"          # update existing entry
draft_to_cfbd["North Alabama"] = None                    # drop — recent FCS-to-FBS
draft_to_cfbd["Tarleton State"] = None                   # drop — recent FCS-to-FBS

# Re-apply mapping
draft["school_std"] = draft["school"].map(
    lambda x: draft_to_cfbd.get(x, x) if x in draft_to_cfbd else x
)

# Drop None mappings
draft_fbs = draft[draft["school_std"].notna()].copy()

# Verify
cfbd_teams = set(recruiting["team"].dropna().str.strip().unique())
draft_schools_std = set(draft_fbs["school_std"].dropna().str.strip().unique())

still_missing = sorted(draft_schools_std - cfbd_teams)
matched = sorted(draft_schools_std & cfbd_teams)

print(f"Matched schools: {len(matched)}")
print(f"Still unmatched: {len(still_missing)}")
print(f"Draft picks remaining: {draft_fbs.shape[0]}")

Matched schools: 214
Still unmatched: 23
Draft picks remaining: 4243


Save clean draft file

In [9]:
draft_fbs.to_csv("data/draft_clean.csv", index=False)
print("Saved draft_clean.csv")
print(draft_fbs[["draft_year", "school_std", "round", "overall", "player_name", "position"]].head(5).to_string())

Saved draft_clean.csv
      draft_year school_std  round  overall       player_name       position
7599        2005       Utah    1.0      1.0        Alex Smith    Quarterback
7600        2005     Auburn    1.0      2.0      Ronnie Brown   Running Back
7601        2005   Michigan    1.0      3.0   Braylon Edwards  Wide Receiver
7602        2005      Texas    1.0      4.0     Cedric Benson   Running Back
7603        2005     Auburn    1.0      5.0  Carnell Williams   Running Back


### Weighted Draft Score
Giving a weighted raft score:
Round 1 pick = 7
Round 2 = 6
Round 3 = 5
Round 4 = 4
Round 5 = 3
Round 6 = 2
Round 7 = 1

In [10]:
import pandas as pd

draft = pd.read_csv("data/draft_clean.csv")

# ── Round weights ────────────────────────────────────────────
round_weights = {1: 7, 2: 5, 3: 4, 4: 3, 5: 2, 6: 1, 7: 1}

draft["round"] = draft["round"].astype(int)
draft["weight"] = draft["round"].map(round_weights)

# ── Collapse to school-year level ────────────────────────────
draft_scored = (
    draft
    .groupby(["draft_year", "school_std"])
    .agg(
        total_picks=("player_name", "count"),
        weighted_draft_score=("weight", "sum"),
        first_round_picks=("round", lambda x: (x == 1).sum()),
        second_round_picks=("round", lambda x: (x == 2).sum()),
    )
    .reset_index()
    .rename(columns={"draft_year": "year", "school_std": "team"})
)

# ── Add zero rows for schools with no draft picks that year ──
# Every FBS school should appear every year, even with 0 picks
all_years = range(draft["draft_year"].min(), draft["draft_year"].max() + 1)
all_teams = draft["school_std"].unique()

full_index = pd.MultiIndex.from_product(
    [all_years, all_teams], names=["year", "team"]
)
draft_scored = (
    draft_scored
    .set_index(["year", "team"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

print(draft_scored.shape)
print(draft_scored.head(10).to_string())
print()

# Sanity check — Alabama should have lots of picks
print("Alabama draft scores by year:")
print(draft_scored[draft_scored["team"] == "Alabama"].to_string())

(4029, 6)
   year            team  total_picks  weighted_draft_score  first_round_picks  second_round_picks
0  2005            Utah            5                    14                  1                   0
1  2005          Auburn            5                    29                  4                   0
2  2005        Michigan            3                    19                  2                   1
3  2005           Texas            3                    15                  2                   0
4  2005   West Virginia            3                    13                  1                   0
5  2005  South Carolina            3                     9                  1                   0
6  2005      Miami (FL)            5                    21                  1                   1
7  2005             USC            5                    25                  2                   2
8  2005            Troy            1                     7                  1                   0
9  2005   

In [11]:
draft_scored.to_csv("data/draft_scored.csv", index=False)
